# Intro to Classes

If not using classes, how data management happens:

In [1]:
account = {"owner":"1", "balance":1000}


def add_balance(acc, amount):
    acc["balance"] += amount


print(account)
add_balance(account,100)
print(account)

{'owner': '1', 'balance': 1000}
{'owner': '1', 'balance': 1100}


In [2]:
def withdraw_balance(acc, amount):
    if acc["balance"] < amount:
        raise ValueError("insufficient balance")
    else:
        acc["balance"] -= amount

print(account)
withdraw_balance(account,100)
print(account)
try:
    withdraw_balance(account,5000)
except:
    print("Error was thrown")
print(account)

{'owner': '1', 'balance': 1100}
{'owner': '1', 'balance': 1000}
Error was thrown
{'owner': '1', 'balance': 1000}


The main issue is there is nothing connecting the function and the data layers, leading to possible data mutation.

We may also need to add data validations in place.

In [3]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def add_balance(self, amount):
        self.balance += amount

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("Insufficient Funds")
        else:
            self.balance -= amount

acc = Account("1", 1000)
acc.add_balance(100)
acc.withdraw(500)
print(acc.balance)

600


In [4]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        self.area = 0
        self.perimeter = 0

    def area(self):
        self.area = self.width * self.height

    def perimeter(self):
        self.perimeter = 2*(self.width + self.height)

## Instance Attributes vs Class Attributes

* An instance attribute lives on one object.
* A class attrib lives on the class and is shared by all instances.
* First the instance is checked, then the class is checked, then base classes.
* Class attribute can be accessed by Class_name.instance.
* If you set a class attrib using instance name then it creates a new instance attr that shadows the class attr (valid for that instance only).
* You should not have mutable types as class instances. Otherwise any changes by any instance will leak to other instances as well.
**Rule of thumb:** class attributes are fine for constants and defaults; never use a mutable object (list/dict/set) as a class attribute unless you *want* it shared.

In [5]:
class Dog:
    species = "Canis familiaris"

    def __init__(self, name):
        self.name = name

a = Dog("Rex")
b = Dog("Fido")

print(a.species, b.species)
print(a.name, b.name)

Canis familiaris Canis familiaris
Rex Fido


In [6]:
# Leakage example:

class Portfolio:
    holdings = []

    def __init__(self):
        pass

    def add(self, ticker):
        self.holdings.append(ticker)


p1 = Portfolio()
p2 = Portfolio()

p1.add("INFY")
print(p1.holdings)
print(p2.holdings)

['INFY']
['INFY']


In [7]:
# Fix for the above:

class Portfolio:
    def __init__(self):
        self.holdings = []

    def add(self, ticker):
        self.holdings.append(ticker)

p1 = Portfolio()
p2 = Portfolio()

p1.add("INFY")
print(p1.holdings)
print(p2.holdings)

['INFY']
[]


### Exercise:

Create a `Counter` class where each instance counts its own increments, but the class

also tracks how many `Counter` objects have ever been created (across all instances).

In [8]:
class Counter:
    total_objects = 0

    def __init__(self):
        self.count_ = 0
        Counter.total_objects += 1
        """Note we write `Counter.total_created += 1`, not `self.total_created += 1`. 
        The latter would read the class value but *write* a new instance attribute, breaking the tally.
        """
    def count(self, add_count):
        self.count_ += add_count


a = Counter()
print(a.count_, a.total_objects)
a.count(1)
print(a.count_, a.total_objects)
b = Counter()
print(a.count_, b.count_, a.total_objects, b.total_objects, Counter.total_objects)

0 1
1 1
1 0 2 2 2


### Types of Methods

* Instance: needs a specific object. The default.
* Class: receives the class (`cls`). 
* Static: a plain function that logically belongs to the class but needs
  neither the instance nor the class. Use it for helpers.

In [9]:
class Temperature:

    def __init__(self, degrees):
        self.degrees = degrees

    # Instance method
    def to_fahrenheit(self):
        return self.degrees*9/5 + 32

    # Class method: receives the class as `cls`
    @classmethod
    def from_fahrenheit(cls, f):
        return cls((f-32)*5/9)

    # Static Method - receives nothing special; just namespaced under the class
    @staticmethod
    def is_freezing(celsius):
        return celsius <= 0

    @classmethod
    def from_kelvin(cls, k):
        return cls(k-273.15)

    @staticmethod
    def valid_celsius(celsius):
        return celsius >= -273.15


t = Temperature(25)
print(t.to_fahrenheit(), type(t))

t2 = Temperature.from_fahrenheit(77)
print(t2.degrees, type(t2))

print(Temperature.is_freezing(-4), type(Temperature))

77.0 <class '__main__.Temperature'>
25.0 <class '__main__.Temperature'>
True <class 'type'>


## Encapsulation: public, protected, private

Encapsulation in Python is a social contract, not a lock. The tool that

Lets you guard attribute access properly is the **property**.

In [10]:
class BankAccount:
    def __init__(self, balance):
        self.balance = balance  #public
        self._pin = "0000"      #internal: don't touch. may change, use at your own risk
        self.__ledger = []      #weakly private. name mangling. Python renames __ledger to _BankAccount__ledger

In [11]:
acc = BankAccount(100)
acc.balance, acc._pin, acc._BankAccount__ledger

(100, '0000', [])

# Properties

* A property lets a method act as an attribute. 
* This lets us start with a plain attribute and then add validation without changing any calling code.
* @property turns radius() into something you access as c.radius (no parens).
* @radius.setter defines what happens on c.radius = value.
* Omit the setter → the property is read-only (like area).
* The real data lives in a differently-named attribute (_radius) to avoid infinite recursion.
* When to use a property vs a plain attribute: start with a plain attribute. Promote it to a property the moment you need validation, a computed value, logging, or lazy-loading. Because the syntax is identical, callers never notice.

In [12]:
import math
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def radius(self):               #defines the getter
        return self._radius

    @radius.setter                  #defines the setter
    def radius(self, value):
        if value <= 0:
            raise ValueError("radius must be positive")
        self._radius = value

    @property                       # read-only computed attribute
    def area(self):     
        return math.pi*self._radius**2

c = Circle(4)
print(c.radius)
print(c.area)
c.radius = 10
print(c.radius)
print(c.area)

4
50.26548245743669
10
314.1592653589793


In [13]:
class Stock:
    def __init__(self, symbol, price):
        self.price = price
        self.symbol = symbol

    @property
    def price(self):
        return self._price

    @price.setter
    def price(self, value):
        if value <= 0:
            raise ValueError("Price cannot be negative")
        self._price = value

    @property
    def is_expensive(self):
        return self._price > 1000



## Inheritance and super()

* Overriding = defining a method with the same name; the child's wins.
* super().__init__(name) — call the parent's version. Always use super() rather than Animal.__init__(self, name); it's what makes multiple inheritance work.
* describe() (defined once in Animal) calls self.speak(), which resolves to whichever subclass's version — the parent code drives child behavior. -- this is called Polymorphism

In [14]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return "..."

    def describe(self):
        return f"{self.name} says {self.speak()}"


class Dog(Animal):
    def speak(self):                # override
        return "Woof"


class Cat(Animal):
    def __init__(self, name, indoor):
        super().__init__(name)      # call Animal.__init__ to set self.name
        self.indoor = indoor

    # def speak(self):
    #     return "Meow"


c = Dog("Rex")
print(c.name, c.speak(), c.describe())


d = Cat("Milo", True)
print(d.name, d.speak(), d.describe())


Rex Woof Rex says Woof
Milo ... Milo says ...


### isinstance and issubclass

In [15]:
d = Dog("Rex")

print(isinstance(d, Dog), issubclass(Dog, Animal), isinstance(d, Animal))

True True True


# Multiple Inheritance and MRO

* Method Resolution Order (MRO): Used to resolve subclasses.


In [16]:
class A:
    def greet(self): return "A"

class B(A):
    def greet(self): return "B -->" + super().greet()

class C(A):
    def greet(self): return "C -->" + super().greet()

class D(B,C):
    def greet(self): return "D -->" + super().greet()

print(D().greet())
print([cls.__name__ for cls in D.__mro__])

D -->B -->C -->A
['D', 'B', 'C', 'A', 'object']


In [17]:
class Vehicle:
    def __init__(self, wheels):
        self.wheels = wheels

    def describe(self):
        return f"Vehicle with {self.wheels} wheels"


class Car(Vehicle):
    def __init__(self, brand):
        super().__init__(4)
        self.brand = brand

    def describe(self):
            return f"Car of brand {self.brand} with {self.wheels} wheels"

class Bike(Vehicle):
    def __init__(self):
        super().__init__(2)


print(Car("Vol").describe())
print(Bike().describe())

Car of brand Vol with 4 wheels
Vehicle with 2 wheels


## Dunder methods - make objects behave like builtins

* These methods let us integrate with Python's syntax. 

\_\_repr__ vs \_\_str__

* Rule: always define __repr__. Define __str__ only if users need a prettier form. If only __repr__ exists, str() falls back to it.

In [18]:
class Money:
    def __init__(self, rupees):
        self.rupees = rupees

    def __repr__(self):
        # for developers. Output: Valid Python code to recreate the object.
        # Calling: repr(obj) or typing the object in a REPL/shell.
        return f"Money({self.rupees!r})"

    def __str__(self):
        # for users
        # Output: A friendly text description
        # Calling: str(obj) or print(obj).
        return f"₹{self.rupees:,.2f}"


m = Money(2.5)
print(m) # looks for __str__
print(repr(m)) # looks for __repr__

₹2.50
Money(2.5)


## Equality vs Ordering



In [19]:
from functools import total_ordering

@total_ordering
class Version:
    def __init__(self, major, minor):
        self.major = major
        self.minor = minor

    def __eq__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return (self.major, self.minor) == (other.major, other.minor)

    def __lt__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return (self.major, self.minor) < (other.major, other.minor)

    def __hash__(self):
         return hash((self.major, self.minor))

    def __repr__(self):
        return f"Version{self.major}, {self.minor}"


v1 = Version(1,2)
v2 = Version(1,2)
v3 = Version(1,3)

print(v1 == v2)
print(v1 < v3)
print(sorted([v1, v3]))

True
True
[Version1, 2, Version1, 3]


In [20]:
class Adder:
    def __init__(self, n): 
        self.n = n

    def __call__(self, x): # makes the INSTANCE callable like a function
        return x + self.n

  
add5 = Adder(5)
print(add5(10)) # 15    -> Adder.__call__(add5, 10)
print(callable(add5)) # True

15
True


In [21]:
class Fraction:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        return (self.x + other.x, self.y + other.y)

    def __eq__(self, other):
        return self.x, self.y == other.x, other.y

    def __repr__(self):
        return f"Fraction{self.x}, {self.y}"

    def __str__(self):
        return f"{self.x}/{self.y}"

    @property
    def y(self):
        return self._y

    @y.setter
    def y(self, value):
        if value == 0:
            raise ValueError("denominator cannot be 0")
        self._y = value



# Abstract Base Classes (ABCs)

ABCs define an interface that subclasses must implement. These can't be instantiated.
Use ABCs when you're designing a framework where plugins/strategies must conform to a contract. They turn "you forgot a method" from a runtime surprise into an immediate,

clear error.

In [22]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self):
        ...

    @abstractmethod
    def perimeter(self):
        ...

    def describe(self):
        return f"area={self.area():.2f} and perimeter={self.perimeter():.2f}"


class Square(Shape):
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side**2

    def perimeter(self):
        return self.side*4


print(Square(3).describe())

area=9.00 and perimeter=12.00


In [27]:
class Strategy(ABC):
    @abstractmethod
    def generate_signal(self, price):
        ...


class BuyTheDip(Strategy):
    def __init__(self, price):
        self.price = price

    def generate_signal(self, price):
        if price < 100:
            return "BUY"
        else: 
            return "HOLD"

class Momentum(Strategy):
    def __init__(self, price):
        self.price = price

    def generate_signal(self, price):
        if price > 100:
            return "BUY"
        else:
            return "SELL"


for strat in [BuyTheDip(100), Momentum(100)]:
    print(type(strat).__name__, strat.generate_signal(100))

BuyTheDip HOLD
Momentum SELL


# DataClasses

* For every data holding class, we dont need to keep writing \_\_init__, \_\_repr__ and \_\_eq__. Similar to pydantic, we can use type annotated fields with @dataclass.
* In the following example: why are we using field(list): if we simply use a list then its shared across all instances. This is not something we want to happen. default_factory runs the callable fresh for every instance.
* Reach for a dataclass whenever a class is mostly a bundle of fields. Reach for a regular class when behavior dominates.
* we can freeze the instances: cannot be modified after creation.

In [30]:
from dataclasses import dataclass, field

@dataclass
class Trade:
    symbol: str
    qty: int
    price: float
    tags: list = field(default_factory=list)


    @property
    def value(self):
        return self.qty*self.price


t = Trade("INFY", 100, 80)
print(t)
print(t.value)

t = Trade("INFY", 100, 80, ["A"])
print(t)
print(t.value)

Trade(symbol='INFY', qty=100, price=80, tags=[])
8000
Trade(symbol='INFY', qty=100, price=80, tags=['A'])
8000


In [40]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Rectangle2:
    length: float
    breadth: float

    @property
    def area(self):
        return self.length*self.breadth

    @property
    def perimeter(self):
        return 2*(self.length+self.breadth)


r = Rectangle2(3,4)
print(r.area, {r})
# r.length = 5: error due to frozen

12 {Rectangle2(length=3, breadth=4)}
